In [1]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

In [2]:
# 2.1 Summarize the dataset by displaying the column data types, counts, and missing values

summary = pd.DataFrame({
    'DataType': df.dtypes,
    'Non-Null Count': df.count(),
    'Missing Values': df.isnull().sum()
})

print(summary)

                    DataType  Non-Null Count  Missing Values
ResponseId             int64           65437               0
MainBranch            object           65437               0
Age                   object           65437               0
Employment            object           65437               0
RemoteWork            object           54806           10631
...                      ...             ...             ...
JobSatPoints_11      float64           29445           35992
SurveyLength          object           56182            9255
SurveyEase            object           56238            9199
ConvertedCompYearly  float64           23435           42002
JobSat               float64           29126           36311

[114 rows x 3 columns]


In [3]:
# 2.2 Generate basic statistics for numerical columns

print(df.describe())

         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000         0.00000   
25%          0.000000 

In [4]:
# 3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country)

print("Number of unique countries:", df['Country'].nunique())
print(df['Country'].value_counts().head(20))
print("Missing values in Country:", df['Country'].isnull().sum())

Number of unique countries: 185
Country
United States of America                                11095
Germany                                                  4947
India                                                    4231
United Kingdom of Great Britain and Northern Ireland     3224
Ukraine                                                  2672
France                                                   2110
Canada                                                   2104
Poland                                                   1534
Netherlands                                              1449
Brazil                                                   1375
Italy                                                    1341
Australia                                                1260
Spain                                                    1123
Sweden                                                   1016
Russian Federation                                        925
Switzerland                   

In [5]:
# 3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format

country_mapping = {
    'USA': 'United States of America',
    'U.S.A.': 'United States of America',
    'United States': 'United States of America',
    'UK': 'United Kingdom',
    'U.K.': 'United Kingdom'
}

edlevel_mapping = {
    'Bachelor’s degree': 'Bachelor’s degree',
    "Bachelor's degree": 'Bachelor’s degree',
    'Master’s degree': 'Master’s degree',
    "Master's degree": 'Master’s degree'
}

df['Country'] = df['Country'].replace(country_mapping)
df['EdLevel'] = df['EdLevel'].replace(edlevel_mapping)

print(df[['Country', 'EdLevel']].head())

                                             Country  \
0                           United States of America   
1  United Kingdom of Great Britain and Northern I...   
2  United Kingdom of Great Britain and Northern I...   
3                                             Canada   
4                                             Norway   

                                             EdLevel  
0                          Primary/elementary school  
1       Bachelor’s degree (B.A., B.S., B.Eng., etc.)  
2    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)  
3  Some college/university study without earning ...  
4  Secondary school (e.g. American high school, G...  


In [6]:
# 4.1 Encode the Employment column using one-hot encoding

employment_encoded = pd.get_dummies(df['Employment'], prefix='Employment')
df = pd.concat([df, employment_encoded], axis=1)

print(employment_encoded.head())

   Employment_Employed, full-time  \
0                            True   
1                            True   
2                            True   
3                           False   
4                           False   

   Employment_Employed, full-time;Employed, part-time  \
0                                              False    
1                                              False    
2                                              False    
3                                              False    
4                                              False    

   Employment_Employed, full-time;Independent contractor, freelancer, or self-employed  \
0                                              False                                     
1                                              False                                     
2                                              False                                     
3                                              False                      

In [7]:
# 5.1 Identify columns with the highest number of missing values

missing_counts = df.isnull().sum().sort_values(ascending=False)
print(missing_counts.head(10))

AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64


In [8]:
# 5.2 Impute missing values in numerical columns (e.g., ConvertedCompYearly) with the median

df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].median())

print("Missing values in ConvertedCompYearly:", df['ConvertedCompYearly'].isnull().sum())

Missing values in ConvertedCompYearly: 0


In [9]:
# 5.3 Impute missing values in categorical columns (e.g., RemoteWork) with the most frequent value

remote_mode = df['RemoteWork'].mode()[0]
df['RemoteWork'] = df['RemoteWork'].fillna(remote_mode)

print("Missing values in RemoteWork:", df['RemoteWork'].isnull().sum())

Missing values in RemoteWork: 0


In [10]:
# 6.1 Apply Min-Max Scaling to normalize the ConvertedCompYearly column

min_val = df['ConvertedCompYearly'].min()
max_val = df['ConvertedCompYearly'].max()

df['ConvertedCompYearly_MinMax'] = (df['ConvertedCompYearly'] - min_val) / (max_val - min_val)

print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())

   ConvertedCompYearly  ConvertedCompYearly_MinMax
0              65000.0                    0.003998
1              65000.0                    0.003998
2              65000.0                    0.003998
3              65000.0                    0.003998
4              65000.0                    0.003998


In [11]:
# 6.2 Log-transform the ConvertedCompYearly column to reduce skewness

import numpy as np

df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly'])

print(df[['ConvertedCompYearly', 'ConvertedCompYearly_Log']].head())

   ConvertedCompYearly  ConvertedCompYearly_Log
0              65000.0                11.082158
1              65000.0                11.082158
2              65000.0                11.082158
3              65000.0                11.082158
4              65000.0                11.082158


In [12]:
# 7.1 Create a new column ExperienceLevel based on the YearsCodePro column

def categorize_experience(value):
    if pd.isna(value):
        return 'Unknown'
    if value == 'Less than 1 year':
        years = 0.5
    elif value == 'More than 50 years':
        years = 51
    else:
        years = float(value)

    if years < 3:
        return 'Beginner'
    elif years < 7:
        return 'Intermediate'
    else:
        return 'Experienced'

df['ExperienceLevel'] = df['YearsCodePro'].apply(categorize_experience)

print(df[['YearsCodePro', 'ExperienceLevel']].head(10))

  YearsCodePro ExperienceLevel
0          NaN         Unknown
1           17     Experienced
2           27     Experienced
3          NaN         Unknown
4          NaN         Unknown
5          NaN         Unknown
6            7     Experienced
7          NaN         Unknown
8          NaN         Unknown
9           11     Experienced
